# TP4 - Optimization : the benefits of momentum and adaptive step-sizes

The goal of this practical is to understand **why** we use the SGD+Momentum and Adam optimizers in deep learning, i.e. why we use momentum and adaptive step sizes, and when they help. We will:

1. test the optimizers on convex quadratics in 2D (least-squares linear regression).,
2. test them on "pathological" 2D loss surfaces (ravines, steep optima), which are specifically designed to show the benefits of momentum and adaptivity,
3. compare them on small neural networks.

Everything can be run on a CPU, no need for a GPU for this practical (even for the neural networks at the end).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm
%matplotlib inline

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, TensorDataset, DataLoader
from torchvision.datasets import MNIST

plt.rcParams.update({"figure.dpi": 110, "font.size": 11, "axes.spines.top": False, "axes.spines.right": False})

def set_seed(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)

set_seed(42)

### Plotting helper functions

In [ ]:
def plot_dataset(dataset : TensorDataset, ax : plt.Axes = None, show_eigenvectors : bool = True):
    """
    Plot the point cloud of the features x_i in R^2 of a dataset. The points are colored by the target y_i
    if the targets are not all equal. If `show_eigenvectors` is True, the eigenvectors u_1, u_2 of the
    empirical covariance H = X^T X / n are drawn from the origin, with length 2 sqrt(lambda_i).
    """
    X, y = dataset.tensors
    X, y = X.numpy(), y.numpy()
    if ax is None:
        _, ax = plt.subplots(figsize=(4, 4))

    if np.allclose(y, y[0]):
        ax.scatter(X[:, 0], X[:, 1], s=12, alpha=0.6, color="tab:blue")
    else:
        sc = ax.scatter(X[:, 0], X[:, 1], c=y, s=12, cmap="viridis")
        plt.colorbar(sc, ax=ax, label="$y$")

    if show_eigenvectors:
        eigvals, eigvecs = np.linalg.eigh(X.T @ X / len(X))
        for k, name in [(1, "$u_1$"), (0, "$u_2$")]:          # eigh sorts the eigenvalues in increasing order
            v = 2 * np.sqrt(eigvals[k]) * eigvecs[:, k]
            ax.annotate("", xy=v, xytext=(0, 0), arrowprops=dict(arrowstyle="-|>", color="tab:red" if k == 1 else "tab:green", lw=2))
            ax.text(*(1.15 * v), name, color="tab:red" if k == 1 else "tab:green", ha="center", va="center", fontsize=12)

    ax.set_xlabel("$x_1$"); ax.set_ylabel("$x_2$"); ax.set_aspect("equal")
    return ax


def plot_mse_contour(dataset : TensorDataset, lim : float = 1.0, ax : plt.Axes = None,
                     n_levels : int = 10, n_grid : int = 200, trajectories : list = None):
    """
    Plot the contour lines of the MSE loss  theta -> 1/(2n) sum_i (y_i - x_i^T theta)^2  of a dataset with
    2 features, on the square [theta_star - lim, theta_star + lim]^2, where theta_star is the minimizer
    (black star). Optionally, `trajectories` is a list of (array of shape (T+1, 2), label) that are drawn on top
    of the contour lines (e.g. the iterates of an optimizer).
    """
    X, y = dataset.tensors
    theta_star = torch.linalg.lstsq(X, y.unsqueeze(1)).solution.squeeze(1)
    c1, c2 = theta_star.tolist()

    t1, t2 = torch.meshgrid(torch.linspace(c1 - lim, c1 + lim, n_grid),
                            torch.linspace(c2 - lim, c2 + lim, n_grid), indexing="xy")
    thetas = torch.stack([t1, t2], dim=-1)                      # (n_grid, n_grid, 2)
    loss = 0.5 * ((thetas @ X.T - y) ** 2).mean(dim=-1)         # MSE at each grid point
    loss_star = 0.5 * ((X @ theta_star - y) ** 2).mean()

    if ax is None:
        _, ax = plt.subplots(figsize=(4, 4))
    levels = loss_star + (loss.max() - loss_star) * torch.logspace(-3, 0, n_levels)
    ax.contour(t1.numpy(), t2.numpy(), loss.numpy(), levels=levels.numpy(), cmap="Blues_r", linewidths=0.8)
    ax.plot(c1, c2, "k*", ms=11, zorder=5)

    for traj, label in (trajectories or []):
        traj = np.asarray(traj)
        ax.plot(traj[:, 0], traj[:, 1], "-o", ms=2.5, lw=1.0, label=label)
        ax.plot(*traj[0], "ko", ms=5, zorder=5)
    if trajectories:
        ax.legend()

    ax.set_xlim(c1 - lim, c1 + lim); ax.set_ylim(c2 - lim, c2 + lim); ax.set_aspect("equal")
    ax.set_xlabel(r"$\theta_1$"); ax.set_ylabel(r"$\theta_2$")
    return ax

def plot_run(dataset, thetas, lim=1.5):
    X, _ = dataset.tensors
    eigvals, eigvecs = torch.linalg.eigh(X.T @ X / len(X))
    lam_1, lam_2, u_1, u_2 = eigvals[1].item(), eigvals[0].item(), eigvecs[:, 1], eigvecs[:, 0]

    fig, axs = plt.subplots(1, 2, figsize=(8.4, 4.2), squeeze=False)
    plot_mse_contour(dataset, lim=lim, ax=axs[0,0], trajectories=[(thetas.numpy(), "")])
    axs[0,0].get_legend().remove()

    axs[0,1].plot((thetas @ u_1).numpy(), color="tab:red", label=rf"$u_1^\top \theta_t$ ($\lambda_1={lam_1:.3g}$)")
    axs[0,1].plot((thetas @ u_2).numpy(), color="tab:purple", label=rf"$u_2^\top\theta_t$ ($\lambda_2={lam_2:.3g}$)")
    axs[0,1].axhline(0, color="k", lw=0.6)
    axs[0,1].set_yscale("symlog", linthresh=1e-3)
    axs[0,1].set_xlabel("iteration $t$")
    axs[0,1].legend(fontsize=8, loc="lower left")
    plt.tight_layout()
    plt.show()


def plot_curve(curve_fn, x_range=(-5, 5), y_range=(-5, 5), plot_3d=False, cmap=cm.viridis, title=""):
    fig = plt.figure()
    ax = plt.axes(projection='3d') if plot_3d else plt.axes()

    x = torch.arange(x_range[0], x_range[1], (x_range[1] - x_range[0]) / 100.)
    y = torch.arange(y_range[0], y_range[1], (y_range[1] - y_range[0]) / 100.)
    x, y = torch.meshgrid(x, y, indexing='xy')
    z = curve_fn(x, y)
    x, y, z = x.numpy(), y.numpy(), z.numpy()

    if plot_3d:
        ax.plot_surface(x, y, z, cmap=cmap, linewidth=1, antialiased=False)
        ax.set_zlabel("loss")
    else:
        ax.imshow(z[::-1], cmap=cmap, extent=(x_range[0], x_range[1], y_range[0], y_range[1]))
    plt.title(title)
    ax.set_xlabel(r"$w_1$")
    ax.set_ylabel(r"$w_2$")
    plt.tight_layout()
    return ax

def plot_lr_sweeps(sweeps):
    fig, axs = plt.subplots(1, 2, figsize=(10, 3.6))
    for name, (lrs, train_losses, val_accs) in sweeps.items():
        axs[0].plot(lrs, train_losses, "o-", label=name)
        axs[1].plot(lrs, val_accs, "o-", label=name)
    for ax in axs:
        ax.set_xscale("log"); ax.set_xlabel("learning rate")
    axs[0].set_yscale("log"); axs[0].set_ylabel("final training loss"); axs[1].set_ylabel("final validation accuracy"); axs[0].legend()
    plt.tight_layout(); plt.show()

## Part 1 - Convex quadratic functions

We now try to understand how SGD, SGD+M and Adam behave on the simplest possible functions to optimize: convex quadratics. While deep learning losses are far from being quadratic, things that are hard in quadratic optimization are also hard when training neural networks. These toy problems will give us explicit reasons for using momentum and adaptive step sizes.

Recall that a convex quadratic optimization problem may be characterized by a Hessian $\mathbf{H} \succeq \mathbf{0}$, which is a $d\times d$ positive semi-definite symmetric matrix, and a minimizer $\theta_\star \in \mathbb{R}^d$, by writing
\begin{equation*}
    f(\theta) = \frac 12 (\theta-\theta_\star)^\top\mathbf{H}(\theta-\theta_\star).
\end{equation*}
This optimization problem occurs in particular in least-squares linear regression, in which, given a dataset $(x_i,y_i) \in \mathbb{R}^d \times \mathbb{R}, i = 1,\dots,n$, one aims to find a linear regressor $\hat\theta$ minimizing
\begin{equation*}
    \min_{\theta \in \mathbb{R}^d} \frac{1}{2n} \sum_{i=1}^n (y_i - x_i^\top \theta)^2.
\end{equation*}
In that case, the Hessian is $\mathbf{H} = \frac{1}{n} \mathbf{X}^\top\mathbf{X}$ (the empirical covariance of the features $x_i$), where $\mathbf{X} \in \mathbb{R}^{n\times d}$ is the feature matrix, whose $i$-th row is $x_i^\top$, and the unique optimum is $\theta_\star = (\mathbf{X}^\top\mathbf{X})^{-1} \mathbf{X}^\top \mathbf{y}$ (if $\mathbf{X}^\top\mathbf{X}$ is non-singular). In deep learning terms, this is a neural network made of a single linear layer of weights $\theta$ (input size $d$, output size $1$), with no activation, trained with the mean-squared error (MSE) loss.

The performance of the optimizers depends strongly on the spectral features of $\mathbf{H}$. Recall that $\mathbf{H}$ has real eigenvalues $\lambda_1 \geq \dots \geq \lambda_d \geq 0$ with an orthonormal basis of eigenvectors $u_1,\dots,u_d$. We are going to see that the **ratios** of the eigenvalues, and the **alignment** of the eigenvectors $u_i$ with the standard basis of $\mathbb R^d$, impact the algorithms. For visualization, we work in 2D ($d=2$).

The function below returns a dataset $\mathbf{X} \in \mathbb R^{n \times 2}$ such that $\mathbf{H} = \frac1n\mathbf X^\top\mathbf X$ has eigenvalues $\lambda_1\geq\lambda_2>0$, and such that its top eigenvector $u_1$ makes an angle of $\rho$ radians with the first standard basis vector $(1,0)^\top$ (so $\rho=0$ means that the eigenvectors are the coordinate axes). We let $\mathbf{y} = 0$ for simplicity: this sets $\theta_\star = 0$, which has no impact on our study, since the optimization problems we study are invariant by translation.

In [ ]:
def generate_dataset(n : int, lambda_1 : float, lambda_2 : float, rho : float) -> Dataset:

    Q, _ = np.linalg.qr(np.random.randn(n, 2), mode="reduced")      # n x 2 matrix with orthonormal columns

    u_1 = np.array([np.cos(rho), np.sin(rho)])
    u_2 = np.array([-np.sin(rho), np.cos(rho)])

    X = np.sqrt(n) * Q @ np.diag(np.sqrt([lambda_1, lambda_2])) @ np.array([u_1, u_2])
    y = np.zeros(n)

    X = torch.tensor(X, dtype=torch.float32)
    y = torch.tensor(y, dtype=torch.float32)

    return TensorDataset(X, y)

We can visualize how $\rho, \lambda_1$ and $\lambda_2$ impact the shape of the loss function. See for yourself how the point cloud and the contour lines change as:
1. $\lambda_1/\lambda_2$ is as small as possible (i.e. equal to $1$),
2. $\lambda_1/\lambda_2$ gets larger,
3. $\rho$ varies.

$\lambda_1/\lambda_2 \geq 1$ is the **condition number**. If it is large, we say that the problem is *ill-conditioned*: the loss has a long, narrow valley, and we will see that the algorithms have a harder time reaching the minimizer.

In [ ]:
n = 200
rho = 1.3
lambda_1 = 1.
lambda_2 = .01

dataset = generate_dataset(n, lambda_1, lambda_2, rho)

fig, axs = plt.subplots(1, 2, figsize=(9, 4))
plot_dataset(dataset, ax=axs[0])
axs[0].set_title("Point cloud of the features")
plot_mse_contour(dataset, lim=2.0, ax=axs[1])
axs[1].set_title("Contour lines of the MSE loss")
plt.tight_layout()
plt.show()

Complete the function `run_optimizer` below, which runs an optimization algorithm given by `optimizer` for a given number of epochs on the least-squares problem defined by a dataloader, starting from $\theta_0$, and returns all the iterates $\theta_0, \theta_1, \dots$. (What you need to implement is basically the usual training loop of PyTorch, with the backward pass and all of that...)

**Careful:** `F.mse_loss` computes $\frac1n\sum_i (y_i-\hat y_i)^2$, **without** the factor $\frac12$ of our loss $\frac{1}{2n}\sum_i (y_i-x_i^\top\theta)^2$. Multiply it by $\frac12$ so that the loss you minimize has Hessian $\mathbf H$ (otherwise it would be $2\mathbf H$, and all the step size thresholds would be divided by two).

In [ ]:
def run_optimizer(model, dataloader, optimizer, n_epochs):
    """
    Run `optimizer` (which must update the parameters of `model`, a linear layer without bias whose weight is the
    parameter theta, already initialized at theta_0) for `n_epochs` epochs on the least-squares problem defined by
    `dataloader`. Returns the iterates theta_t (one per optimizer step, theta_0 included) as a tensor of shape (T + 1, 2).
    """
    
    thetas = [model.weight.detach().clone().flatten()]
    for epoch in range(n_epochs):
        for X_batch, y_batch in dataloader:
            # TODO: compute the predictions of the model, the loss (do not forget the factor 1/2),
            # reset the gradients, compute the gradient of the loss, and do one optimizer step

            thetas.append(model.weight.detach().clone().flatten())

    return torch.stack(thetas)

Write a code that runs $100$ iterations of SGD starting from $\theta_0 = 0.5 u_1 + u_2$ and plots the results with `plot_run`, which takes as argument a `dataset`, and the `thetas` outputed by the `run_optimizer`function. Test this script on various datasets (with varying $\lambda_1,\lambda_2$) and learning rates, in particular learning rates of the form $\eta = c/\lambda_1$ or $c/\lambda_2$, with $c \in (0,2.5)$. 

What do you observe? Does the algorithm diverge/converge, if so in what directions? Does it oscillate? For what values of $c$? Why are ill-conditioned problems especially hard? Does the angle $\rho$ impact the convergence trajectory? Why?

PS: You will need to define the model first, which is simply a `nn.Linear` layer

In [ ]:
## Dataset and Dataloader, test SGD on various parameters lambda_1, lambda_2

n = 200
lambda_1 = 1.
lambda_2 = .01
rho = np.pi/4

dataset = generate_dataset(n, lambda_1, lambda_2, rho)
dataloader = DataLoader(dataset, batch_size = n, shuffle=False)    # Full batch gradient descent -> batch_size = n

u_1 = np.array([np.cos(rho), np.sin(rho)])
u_2 = np.array([-np.sin(rho), np.cos(rho)])

In [ ]:
## TODO: write script to run SGD on the dataset for 100 epochs, with learning rate lr, and plot the run

thetas = ...
plot_run(dataset, thetas)

Your empirical observations for $c$ can actually be proved theoretically: indeed, you can prove (try to do it!) that for gradient descent on the quadratic function, we have for all $t$ and all $i$
\begin{align*}
   \langle u_i , \theta_{t+1} - \theta_\star \rangle = (1 - \eta \lambda_i) \langle u_i , \theta_{t} - \theta_\star \rangle.
\end{align*}
Seeing this, can you give the ranges of $c$ for which the algorithm diverges/converges? The rangles at which it oscillates? In which directions?

Now, lets run the same experiments for SGD with momentum, starting from the same point. Test different values of learning rate, with mometum parameter fixed at $0.9$. Does momentum help? Are the benefits in comparison to SGD more important when the condition number is small or large? Does the angle $\rho$ impact the convergence trajectory? Why?

In [ ]:
## TODO: write script to run SGD+Momentum on the dataset for 100 epochs, with learning rate lr and momentum 0.9, and plot the run

thetas = ...
plot_run(dataset, thetas)

Finally, do the same for Adam. Does step-size adaptivity help? Why? Does the angle $\rho$ impact the convergence trajectory? Why?

In [ ]:
## TODO: write script to run Adam on the dataset for 100 epochs, with learning rate lr and default hyperparameters beta_1,beta_2,epsilon, and plot the run

thetas = ...
plot_run(dataset, thetas)

## Part 2 - Pathological loss surfaces

We now test our optimizers on two 2D loss surfaces that are specifically designed to be difficult for algorithms without mometum and step-size adaptivity. They are taken from the tutorial *Optimization and Initialization* of the UvA Deep Learning course (Phillip Lippe), where you can find more details. In both cases, $w=(w_1,w_2)$ plays the role of the parameters and the surface is the loss.

First, a helper to train 2 parameters on it with one of our optimizers.

In [ ]:
def train_curve(weights, optimizer, loss_fn, num_updates=100):
    """
    Inputs:
        weights - The weight tensor that will be updated during the optimization
        optimizer - Optimizer, with the appropriate parameters to update
        loss_fn - Loss function (e.g. pathological curvature)
        num_updates - Number of updates/steps to take when optimizing
    Outputs:
        Numpy array of shape [num_updates, 3] with [t,:2] being the parameter values at step t, and [t,2] the loss at t.
    """

    list_points = []
    for _ in range(num_updates):
        loss = loss_fn(weights[0], weights[1])
        list_points.append(torch.cat([weights.data.detach(), loss.unsqueeze(dim=0).detach()], dim=0))
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    return torch.stack(list_points, dim=0).numpy()


def plot_runs_on_curve(curve_fn, runs, x_range, y_range, title=""):
    """`runs` is a dictionary {name: output of train_curve}."""
    ax = plot_curve(curve_fn, x_range=x_range, y_range=y_range, plot_3d=False, title=title)
    for (name, points), color in zip(runs.items(), ["red", "blue", "grey"]):
        ax.plot(points[:, 0], points[:, 1], color=color, marker="o", label=name, alpha=0.8)
    ax.set_xlim(x_range); ax.set_ylim(y_range)
    plt.legend(); plt.show()

the first pathological curve is a ravine, where the gradient is steep in one direction (here $w_1$, with the optimum at the center of the ravine), while it is much smaller in a second direction ($w_2$) along which we have to travel to reach the optimum. If we find ourselves on the ridges, the gradient is much larger in $w_1$ than in $w_2$, and we may jump from one side of the ravine to the other. Does it remind you of something from Part 1?

In [ ]:
def ravine_loss(w1, w2):
    x1_loss = torch.tanh(w1)**2 + 0.01 * torch.abs(w1)
    x2_loss = torch.sigmoid(w2)
    return x1_loss + x2_loss

_ = plot_curve(ravine_loss, plot_3d=True, title="Pathological curvature")
plt.show()

Run the three methods (SGD, SGD+M with momentum $0.9$, Adam) for $100$ updates from $w=(5,5)$ with carefully chosen learning rates. The `runs[alg]` should contain the output of the function `train_curve`. To run this function, you will need to define beforehand an optimizer, which will update a 2D tensor `weights`. Make sure `weights` has `requires_grad=True`.

What are the best algorithms for this pathological function? Why?

NB: an `Optimizer` takes as first argument a *list* of tensors.

NB2: for these optimization problems, you may take larger learning rates than in Part 1.

NB3: make sure that for each algorithm, you reset the `weights`, so that they all start at $w=(5,5)$.

In [ ]:
runs = {}

In [ ]:
# TODO: let runs["SGD"] be the output of train_curve for SGD with 100 updates

runs["SGD"] = ...

In [ ]:
# TODO: let runs["SGD+M"] be the output of train_curve for SGD with momentum with 100 updates

runs["SGD+M"] = ...

In [ ]:
# TODO: let runs["Adam"] be the output of train_curve for Adam with 100 updates

runs["Adam"] = ...

In [ ]:
all_points = np.concatenate(list(runs.values()), axis=0)
plot_runs_on_curve(ravine_loss, runs, x_range=(-np.abs(all_points[:, 0]).max(), np.abs(all_points[:, 0]).max()),
                   y_range=(all_points[:, 1].min(), all_points[:, 1].max()), title="Ravine loss")

This pathological function shows the benefits of momentum, but step-size adaptivity is not necessary to obtain good results. A second type of challenging surfaces is *steep optima*: a large part of the surface has very small gradients, while around the optima the gradients are very large. The surface below is a sum of three narrow Gaussian wells.

In [ ]:
def bivar_gaussian(w1, w2, x_mean=0.0, y_mean=0.0, x_sig=1.0, y_sig=1.0):
    norm = 1 / (2 * np.pi * x_sig * y_sig)
    x_exp = (-1 * (w1 - x_mean)**2) / (2 * x_sig**2)
    y_exp = (-1 * (w2 - y_mean)**2) / (2 * y_sig**2)
    return norm * torch.exp(x_exp + y_exp)

def wells_func(w1, w2):
    z = -bivar_gaussian(w1, w2, x_mean=1.0, y_mean=-0.5, x_sig=0.2, y_sig=0.2)
    z -= bivar_gaussian(w1, w2, x_mean=-1.0, y_mean=0.5, x_sig=0.2, y_sig=0.2)
    z -= bivar_gaussian(w1, w2, x_mean=-0.5, y_mean=-0.8, x_sig=0.2, y_sig=0.2)
    return z

_ = plot_curve(wells_func, x_range=(-2, 2), y_range=(-2, 2), plot_3d=True, title="Steep optima")
plt.show()

Similarly as before, fill the `runs` dictionary and plot the results. Do $50$ iterations and start from $w = (0,0)$. What do you observe? Why do the non-adaptive algorithms struggle?

In [ ]:
runs = {}

In [ ]:
# TODO: let runs["SGD"] be the output of train_curve for SGD with 50 updates

runs["SGD"] = ...

In [ ]:
# TODO: let runs["SGD+M"] be the output of train_curve for SGD with momentum with 50 updates

runs["SGD+M"] = ...

In [ ]:
# TODO: let runs["Adam"] be the output of train_curve for Adam with 50 updates

runs["Adam"] = ...

In [ ]:
all_points = np.concatenate(list(runs.values()), axis=0)
plot_runs_on_curve(wells_func, runs, x_range=(-np.abs(all_points[:, 0]).max(), np.abs(all_points[:, 0]).max()),
                   y_range=(all_points[:, 1].min(), all_points[:, 1].max()), title="Wells loss")

## Part 3 - Small neural networks

We finally compare the optimizers to train a small neural network (a multi-layer perceptron) on MNIST. To keep the experiments fast on a laptop, we use $10{,}000$ training images and train for $5$ epochs with batches of size $128$. All the images are flattened into vectors of size $784$, and normalized.

In [ ]:
DATASET_PATH = "data"

train_full = MNIST(root=DATASET_PATH, train=True, download=False)       # If you already have MNIST on your laptop, you don't need to download it again, just move it into the right directory
X_all = train_full.data.float().view(-1, 784) / 255.
X_all = (X_all - X_all.mean()) / X_all.std()                   # normalize
y_all = train_full.targets

X_train, y_train = X_all[:10000], y_all[:10000]                  # training set (10k images)
X_val, y_val = X_all[50000:], y_all[50000:]                      # validation set (10k images)
print(X_train.shape, X_val.shape)

The function `train_model` trains a fresh MLP with $784$ inputs, one hidden layer of $128$ ReLU units, and $10$ outputs, with the cross-entropy loss, using the optimizer created by `optimizer_fn`. It returns the training loss at each iteration, and the training loss and validation accuracy after each epoch. If the loss becomes `nan` (divergence), the training stops.

The only thing you need to do for this section is to complete the training loop in the `train_model` function, this time using the Cross Entropy Loss.

In [ ]:
def make_mlp(hidden=(128,)):
    layers, d = [], 784
    for h in hidden:
        layers += [nn.Linear(d, h), nn.ReLU()]
        d = h
    return nn.Sequential(*layers, nn.Linear(d, 10))


def train_model(optimizer_fn, X_train, y_train, X_val, y_val, n_epochs=5, batch_size=128, hidden=(128,), seed=0):
    torch.manual_seed(seed)
    model = make_mlp(hidden)
    optimizer = optimizer_fn(model.parameters())
    loader = DataLoader(TensorDataset(X_train, y_train), batch_size=batch_size, shuffle=True)

    history = dict(iter_loss=[], epoch_train_loss=[], epoch_val_acc=[])
    for epoch in range(n_epochs):
        for xb, yb in loader:
            ## TODO : write the training loop.
            ## Make sure the loss variable is named `loss`, since we store it in the history dictionary afterwards 

            history["iter_loss"].append(loss.item())
            if not torch.isfinite(loss):
                return history                                   # diverged: stop the training
        with torch.no_grad():
            history["epoch_train_loss"].append(F.cross_entropy(model(X_train), y_train).item())
            history["epoch_val_acc"].append((model(X_val).argmax(dim=1) == y_val).float().mean().item())
    return history

We can compare the performance of the three optimization algorithms for particular choices of learning rates.

In [ ]:
histories = {}
histories["SGD"] = train_model(lambda params : torch.optim.SGD(params, lr=.1), X_train, y_train, X_val, y_val)
histories["SGD+M"] = train_model(lambda params : torch.optim.SGD(params, lr=.1, momentum=.9), X_train, y_train, X_val, y_val)
histories["Adam"] = train_model(lambda params : torch.optim.Adam(params, lr=1e-3), X_train, y_train, X_val, y_val)

fig, axs = plt.subplots(1, 2, figsize=(10, 3.6))
for name, h in histories.items():
    smooth = np.convolve(h["iter_loss"], np.ones(20) / 20, mode="valid")
    axs[0].plot(smooth, label=name)
    axs[1].plot(range(1, len(h["epoch_val_acc"]) + 1), h["epoch_val_acc"], "o-", label=name)
axs[0].set_yscale("log"); axs[0].set_xlabel("iteration"); axs[0].set_ylabel("training loss"); axs[0].legend()
axs[1].set_xlabel("epoch"); axs[1].set_ylabel("validation accuracy")
plt.tight_layout(); plt.show()

Of course, a fair comparison requires **tuning the learning rate** of each optimizer. The cell below trains the network for each learning rate of a grid, and plots the final training loss and validation accuracy as a function of the learning rate (a value of `nan` means that the training diverged). It takes about a minute.

In [ ]:
def lr_sweep(optimizer_cls, lrs, X_train, y_train, X_val, y_val, **kwargs):
    train_losses, val_accs = [], []
    for lr in lrs:
        h = train_model(lambda p: optimizer_cls(p, lr=lr, **kwargs), X_train, y_train, X_val, y_val)
        diverged = len(h["epoch_train_loss"]) == 0 or not np.isfinite(h["iter_loss"][-1])
        train_losses.append(np.nan if diverged else h["epoch_train_loss"][-1])
        val_accs.append(np.nan if diverged else h["epoch_val_acc"][-1])
    return np.array(train_losses), np.array(val_accs)

lrs_sgd = np.geomspace(1e-3, 1, 7)
lrs_adam = np.geomspace(1e-5, 1e-1, 7)

sweeps = {}
sweeps["SGD"] = (lrs_sgd, *lr_sweep(torch.optim.SGD, lrs_sgd, X_train, y_train, X_val, y_val))
sweeps["SGD+M"] = (lrs_sgd, *lr_sweep(torch.optim.SGD, lrs_sgd, X_train, y_train, X_val, y_val, momentum=0.9))
sweeps["Adam"] = (lrs_adam, *lr_sweep(torch.optim.Adam, lrs_adam, X_train, y_train, X_val, y_val))
plot_lr_sweeps(sweeps)

For this experiment, SGD+M should achieve the best results for the best learning rate choice, which is typically the case for such small neural networks. In Part 1, we saw that Adam is insensitive to ill-conditioning when it is **axis-aligned**. A natural way for ill-conditioning to appear in a neural network is a **badly scaled input**: if some input features are much larger than others, the loss is much more curved along the weights attached to the large features, and flatter along the others, which are coordinate directions. The cell below multiplies each of the $784$ input features by a random scale (log-normal, with a spread of several orders of magnitude), and repeats the learning rate sweep.

In [ ]:
g = torch.Generator().manual_seed(1)
scales = torch.exp(2 * torch.randn(784, generator=g))          # random log-normal feature scales, spanning several orders of magnitude (see min and max below)
print(f"feature scales: min = {scales.min().item():.3g}, max = {scales.max().item():.3g}")

X_train_bad, X_val_bad = X_train * scales, X_val * scales

sweeps_bad = {}
sweeps_bad["SGD"] = (lrs_sgd, *lr_sweep(torch.optim.SGD, lrs_sgd, X_train_bad, y_train, X_val_bad, y_val))
sweeps_bad["SGD+M"] = (lrs_sgd, *lr_sweep(torch.optim.SGD, lrs_sgd, X_train_bad, y_train, X_val_bad, y_val, momentum=0.9))
sweeps_bad["Adam"] = (lrs_adam, *lr_sweep(torch.optim.Adam, lrs_adam, X_train_bad, y_train, X_val_bad, y_val))
plot_lr_sweeps(sweeps_bad)

Here, Adam should significantly outperform the other algorithms, showing the interest of step-size adaptivity for losses with very different scales on different coordinates.